In [1]:
import pandas as pd
import numpy as np
from src import run_experiment
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer, KNNImputer
from src import Config

In [2]:
df = pd.read_csv('../data/app_train_nan90.csv')

In [3]:
# Солвер 'saga' с penalty='elasticnet' очень долго сходится, поэтому сделаем пока чистую L2-регуляризацию
def log_reg_param_space(trial):
    return {
        "C": trial.suggest_float('C', 1e-3, 1e2, log=True),
        # "l1_ratio": trial.suggest_float('l1_ratio', 0, 1, step=0.1),
        # "penalty": trial.suggest_categorical('penalty', ['elasticnet']),
        # "solver": trial.suggest_categorical('solver', ['saga']),
        "max_iter": trial.suggest_int('max_iter', 100, 1000, step=100)
    }

exp_name_lr = f'90_nan'

# run_experiment(
#     exp_name_lr,
#     LogisticRegression,
#     log_reg_param_space,
#     df,
#     n_trials=1,
#     cv_num=2,
#     save_model=True
# )

In [4]:
from sklearn.tree import DecisionTreeClassifier

def decision_tree_param_space(trial):
    return {
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        "max_depth": trial.suggest_int("max_depth", 1, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    }

exp_name_dt = f'90_nan'

# run_experiment(
#     exp_name_dt,
#     DecisionTreeClassifier,
#     decision_tree_param_space,
#     df,
#     n_trials=1,
#     cv_num=2,
#     save_model=True
# )

In [5]:
from sklearn.ensemble import RandomForestClassifier

def random_forest_param_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 1, 13),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False])
    }

exp_name_rf = f'90_nan'

# run_experiment(
#     exp_name_rf,
#     RandomForestClassifier,
#     random_forest_param_space,
#     df,
#     n_trials=1,
#     cv_num=2,
#     save_model=True
# )

In [6]:
from catboost import CatBoostClassifier

def catboost_param_space(trial):
    return {
        "iterations": trial.suggest_int("iterations", 100, 500),
        "depth": trial.suggest_int("depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 0, 1),
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        "bootstrap_type": "Bernoulli",
        "verbose": False,
        "random_state": Config.SEED
    }

exp_name_cb = f'90_nan'

# run_experiment(
#     exp_name_cb,
#     CatBoostClassifier,
#     catboost_param_space,
#     df,
#     n_trials=1,
#     cv_num=2,
#     save_model=True
# )

In [ ]:
from lightgbm import LGBMClassifier

def lightgbm_param_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        # "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        # "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        # "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "objective": "binary",
        "data_sample_strategy": "goss",
        "early_stopping_rounds": 50,
        "eval_metric": "auc",
        "verbose": -1,
        "random_state": Config.SEED
    }

exp_name_lgbm = f'90_nan'

run_experiment(
    exp_name_lgbm,
    LGBMClassifier,
    lightgbm_param_space,
    df,
    n_trials=10,
    cv_num=5,
    save_model=False
)

[I 2026-05-04 23:47:23,844] Using an existing study with name 'LGBMClassifier' instead of creating a new one.
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
/Users/uralgimazov/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/uralgimazov/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   20.3s remaining:   30.5s
/Users/uralgimazov/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/uralgimazov/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:273

# Export to the PROD

Из всех моделей осталось выбрать наилучшую по Kaggle-score и дальше через FastAPI & Streamlit UI сделать полноценное приложение.